# Wave Equation and C Code Generation

This notebook uses current NRPy 2 wave-equation modules to construct Cartesian right-hand sides and exact-solution expressions, then prints small C assignment snippets with `nrpy.c_codegen.c_codegen`.

## Table of Contents

1. [First-order wave system](#First-order-wave-system)
2. [NRPy state hygiene and imports](#NRPy-state-hygiene-and-imports)
3. [Current NRPy wave right-hand sides](#Current-NRPy-wave-right-hand-sides)
4. [Hand-derived residuals](#Hand-derived-residuals)
5. [Exact-solution expressions](#Exact-solution-expressions)
6. [C assignment snippets](#C-assignment-snippets)
7. [Next notebook](#Next-notebook)

## First-Order Wave System

The Cartesian wave equation

$$
\partial_t^2 u = c^2(\partial_x^2 u+\partial_y^2 u+\partial_z^2 u)
$$

is written as

$$
\partial_t u = v,\qquad
\partial_t v = c^2(\partial_x^2 u+\partial_y^2 u+\partial_z^2 u).
$$

NRPy represents derivative variables symbolically, then lowers those symbols to finite-difference memory reads during generated-project construction.

## NRPy State Hygiene and Imports

The next cell clears only the names introduced by this notebook before instantiating the NRPy wave-equation classes.

In [ ]:
import sympy as sp

import nrpy
import nrpy.c_codegen as ccg
import nrpy.grid as gri
import nrpy.indexedexp as ixp
import nrpy.params as par
from nrpy.equations.wave_equation.WaveEquation_RHSs import WaveEquation_RHSs
from nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData import (
    WaveEquation_solution_Cartesian,
)


print("Imported nrpy from:", nrpy.__file__)

existing_wave_gridfunctions = sorted(
    name for name in ["uu", "vv"] if name in gri.glb_gridfcs_dict
)
existing_wave_parameters = sorted(
    name
    for name in ["wavespeed", "time", "sigma", "kk0", "kk1", "kk2"]
    if name in par.glb_params_dict or name in par.glb_code_params_dict
)
print("Existing wave gridfunctions before setup:", existing_wave_gridfunctions or "none")
print("Existing wave parameters before setup:", existing_wave_parameters or "none")

## Current NRPy Wave Right-Hand Sides

`WaveEquation_RHSs` registers `uu` and `vv` if needed, declares the second-derivative symbols, and stores the right-hand sides as SymPy expressions.

In [ ]:
rhs = WaveEquation_RHSs()

print("uu_rhs =", rhs.uu_rhs)
print("vv_rhs =", rhs.vv_rhs)
print("Registered wave gridfunctions:", sorted(name for name in gri.glb_gridfcs_dict if name in {"uu", "vv"}))

## Hand-Derived Residuals

The residuals below compare the module expressions against the direct first-order system. A zero residual means the expressions match after symbolic simplification.

In [ ]:
wavespeed = sp.Symbol("wavespeed", real=True)
vv = sp.Symbol("vv", real=True)
uu_dDD = ixp.declarerank2("uu_dDD", symmetry="sym01")

hand_uu_rhs = vv
hand_vv_rhs = wavespeed**2 * sum(uu_dDD[i][i] for i in range(3))

print("uu residual:", sp.simplify(rhs.uu_rhs - hand_uu_rhs))
print("vv residual:", sp.simplify(rhs.vv_rhs - hand_vv_rhs))

## Exact-Solution Expressions

The Cartesian exact-solution module provides plane-wave and spherical-Gaussian solutions. These expressions are used by generated examples to set initial data and evaluate errors.

In [ ]:
plane_wave = WaveEquation_solution_Cartesian(WaveType="PlaneWave")
spherical_gaussian = WaveEquation_solution_Cartesian(WaveType="SphericalGaussian")

print("Plane-wave u expression:")
print(plane_wave.uu_exactsoln)
print("\nPlane-wave v expression:")
print(plane_wave.vv_exactsoln)
print("\nSpherical-Gaussian fields available:")
for name in sorted(spherical_gaussian.__dict__):
    print(" ", name)

## C Assignment Snippets

These snippets are intentionally small. Full generated projects combine the expressions with grid loops, finite-difference data access, parameter handling, diagnostics, and build files.

In [ ]:
rhs_code = ccg.c_codegen(
    [rhs.uu_rhs, rhs.vv_rhs],
    ["uu_rhs", "vv_rhs"],
    include_braces=False,
    verbose=False,
)
print(rhs_code)

In [ ]:
exact_code = ccg.c_codegen(
    [plane_wave.uu_exactsoln, plane_wave.vv_exactsoln],
    ["uu_exact", "vv_exact"],
    include_braces=False,
    verbose=False,
)
print(exact_code)

## Next Notebook

[Start-to-finish Cartesian wave project](start_to_finish_wave_cartesian.ipynb) runs the package generator, inspects the generated files, builds when local tools are available, and reads convergence diagnostics.